In [58]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import sys
import os

sys.path.append('/Users/danylo/Desktop/coursework/lab3/src')
from ling_features import TextLinguist

DATA_PATH = '/Users/danylo/Desktop/coursework/lab3/data/processed_v2.csv' 
df = pd.read_csv(DATA_PATH)
print(f"Завантажено даних: {len(df)}")

Завантажено даних: 2000


In [59]:
print("Ініціалізація")
linguist = TextLinguist()

print("Обробка текстів")
premise_features = df['premise_v2'].apply(lambda x: linguist.extract_features(str(x)))
df['premise_lemma'] = premise_features.apply(lambda x: x['lemma_text'])
df['premise_pos'] = premise_features.apply(lambda x: x['pos_seq'])

hypothesis_features = df['hypothesis_v2'].apply(lambda x: linguist.extract_features(str(x)))
df['hypothesis_lemma'] = hypothesis_features.apply(lambda x: x['lemma_text'])
df['hypothesis_pos'] = hypothesis_features.apply(lambda x: x['pos_seq'])

df.to_csv('/Users/danylo/Desktop/coursework/lab3/data/processed_v3_with_lemmas.csv', index=False)

Ініціалізація


2026-03-03 12:38:26 INFO: Downloaded file to /Users/danylo/Library/Caches/stanza/1.11.0/resources/resources.json
2026-03-03 12:38:26 WARNING: Language uk package default expects mwt, which has been added
2026-03-03 12:38:26 INFO: Downloading these customized packages for language: uk (Ukrainian)...
| Processor       | Package     |
---------------------------------
| tokenize        | iu          |
| mwt             | iu          |
| pos             | iu_charlm   |
| lemma           | iu_nocharlm |
| backward_charlm | conll17     |
| pretrain        | conll17     |
| forward_charlm  | conll17     |

2026-03-03 12:38:26 INFO: File exists: /Users/danylo/Library/Caches/stanza/1.11.0/resources/uk/tokenize/iu.pt
2026-03-03 12:38:26 INFO: File exists: /Users/danylo/Library/Caches/stanza/1.11.0/resources/uk/mwt/iu.pt
2026-03-03 12:38:26 INFO: File exists: /Users/danylo/Library/Caches/stanza/1.11.0/resources/uk/pos/iu_charlm.pt
2026-03-03 12:38:26 INFO: File exists: /Users/danylo/Library/Cache

Обробка текстів


In [80]:
df['text_raw'] = df['premise_v2'] + " " + df['hypothesis_v2']
df['text_lemma'] = df['premise_lemma'] + " " + df['hypothesis_lemma']
df['text_pos'] = df['premise_pos'] + " " + df['hypothesis_pos'] 

X_raw = df['text_raw']
X_lemma = df['text_lemma']
X_pos = df['text_pos'] 
y = df['labels']

X_raw_train, X_raw_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2)
X_lem_train, X_lem_test, _, _ = train_test_split(X_lemma, y, test_size=0.2)
X_pos_train, X_pos_test, _, _ = train_test_split(X_pos, y, test_size=0.2) 

def train_and_evaluate(X_tr, X_te, y_tr, y_te, name="Baseline", ngram_range=(1,1)):
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=ngram_range)
    X_tr_vec = vectorizer.fit_transform(X_tr)
    X_te_vec = vectorizer.transform(X_te)
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_tr_vec, y_tr)
    preds = clf.predict(X_te_vec)
    
    acc = accuracy_score(y_te, preds)
    f1 = f1_score(y_te, preds, average='macro')
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1: {f1:.4f}\n")
    return acc, f1

print("\nОЦІНКА МОДЕЛЕЙ:")
train_and_evaluate(X_raw_train, X_raw_test, y_train, y_test, name="Baseline 1 (RAW processed_v2)")
train_and_evaluate(X_lem_train, X_lem_test, y_train, y_test, name="Baseline 2 (LEMMA text)")

train_and_evaluate(X_pos_train, X_pos_test, y_train, y_test, name="Baseline 3 (POS tags only)", ngram_range=(1,3))


ОЦІНКА МОДЕЛЕЙ:
--- Baseline 1 (RAW processed_v2) ---
Accuracy: 0.4225
Macro-F1: 0.4225

--- Baseline 2 (LEMMA text) ---
Accuracy: 0.3450
Macro-F1: 0.3447

--- Baseline 3 (POS tags only) ---
Accuracy: 0.3375
Macro-F1: 0.3354



(0.3375, 0.3353869467275179)